## A. Data prep
Chỉ xóa exact duplicates; giữ các dòng trùng business key có metrics khác nhau.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull
from IPython.display import display

pd.options.display.float_format = '{:,.6f}'.format
platforms = ['Google', 'Meta', 'TikTok']
root = Path.cwd() if (Path.cwd() / 'data/raw').exists() else Path.cwd().parent
csv_path = root / 'data/raw/global_ads_performance_dataset.csv'
if not csv_path.exists():
    csv_path = Path('/mnt/data/global_ads_performance_dataset.csv')
df = pd.read_csv(csv_path, parse_dates=['date'])
exact_duplicates = df.duplicated().sum()
df = df.drop_duplicates()
key = ['date', 'platform', 'campaign_type', 'industry', 'country']
print('Exact duplicates removed:', exact_duplicates)
print('Rows in duplicated business keys retained:', df.duplicated(key, keep=False).sum())
df['platform'] = df['platform'].replace({'Google Ads': 'Google', 'Meta Ads': 'Meta', 'TikTok Ads': 'TikTok'})
df = df.loc[df['date'].dt.year.eq(2024) & df['platform'].isin(platforms)].copy()
df['month'] = df['date'].dt.to_period('M')

## B. Platform × Month
Tổng metrics và pooled ratios; mọi quyết định ở platform level.

In [ ]:
metrics = {'ad_spend': 'Spend', 'impressions': 'Impressions', 'clicks': 'Clicks',
           'conversions': 'Conversions', 'revenue': 'Revenue'}
monthly = df.groupby(['month', 'platform'])[list(metrics)].sum().rename(columns=metrics)

def economics(totals):
    out = totals.copy()
    for name, numerator, denominator in [
        ('CPC', 'Spend', 'Clicks'), ('CVR', 'Conversions', 'Clicks'),
        ('RPC', 'Revenue', 'Conversions'), ('ROAS', 'Revenue', 'Spend'),
        ('CPA', 'Spend', 'Conversions'), ('Imp_per_dollar', 'Impressions', 'Spend'),
        ('Click_per_dollar', 'Clicks', 'Spend'), ('Conv_per_dollar', 'Conversions', 'Spend'),
        ('Rev_per_dollar', 'Revenue', 'Spend')]:
        out[name] = out[numerator] / out[denominator]
    return out

monthly = economics(monthly)
spend = monthly['Spend'].unstack('platform').reindex(columns=platforms)
platform_summary = economics(monthly.groupby('platform')[list(metrics.values())].sum())
platform_summary = platform_summary.reindex(platforms)[['Spend', 'CPC', 'CVR', 'RPC', 'ROAS', 'CPA']]

## C. Baseline Q4
Allocation từ pooled Q4 spend; B là tổng spend trung bình mỗi tháng Q4.

In [ ]:
def baseline_for(months):
    recent_spend = spend.loc[months].iloc[-3:]
    return (recent_spend.sum() / recent_spend.to_numpy().sum()).to_numpy(), recent_spend.sum(axis=1).mean()

x0, B = baseline_for(spend.index)
print(f'Monthly simulation budget B: {B:,.2f}')
print('Baseline shares:', dict(zip(platforms, x0)))

## D. Historical economic states
12 tháng 2024 là các state quan sát; không tạo scenario giả định.

In [ ]:
state_names = ['Imp_per_dollar', 'Click_per_dollar', 'Conv_per_dollar', 'Rev_per_dollar']
states = {name: monthly[name].unstack('platform').reindex(columns=platforms) for name in state_names}

## E. Joint historical support
Convex hull trên Google share và Meta share. Tolerance chỉ xử lý sai số số thực.

In [ ]:
actual_shares = spend.div(spend.sum(axis=1), axis=0)
tol = 1e-10

def build_hull(months):
    return ConvexHull(actual_shares.loc[months, ['Google', 'Meta']].to_numpy())

hull = build_hull(spend.index)

def inside_historical_support(g_share, m_share, support=hull):
    points = np.stack(np.broadcast_arrays(g_share, m_share), axis=-1)
    return np.all(points @ support.equations[:, :2].T + support.equations[:, 2] <= tol, axis=-1)

## F. Candidate grid
Grid 1 điểm phần trăm, tổng shares bằng 1; chỉ giữ candidate trong hull.

In [ ]:
grid = np.array([(g, m, 100 - g - m) for g in range(101) for m in range(101 - g)]) / 100
candidates = grid[inside_historical_support(grid[:, 0], grid[:, 1])]
print(f'Total grid: {len(grid)}; inside support: {len(candidates)}')

## G. Historical evaluation
Tính outcome và uplift trên cùng budget B cho candidate và baseline.

In [ ]:
def evaluate(allocations, months, baseline, budget):
    outcomes, uplifts = {}, {}
    for metric, state in [('revenue', 'Rev_per_dollar'), ('conversion', 'Conv_per_dollar'),
                          ('impression', 'Imp_per_dollar')]:
        rates = states[state].loc[months].to_numpy()
        outcomes[metric] = budget * (allocations @ rates.T)
        baseline_outcome = budget * (rates @ baseline)
        uplifts[metric] = outcomes[metric] / baseline_outcome - 1
    return outcomes, uplifts

outcomes, uplifts = evaluate(candidates, spend.index, x0, B)

## H. Core business filters
Không giảm impressions hoặc conversions trong bất kỳ training state nào.

In [ ]:
def funnel_mask(uplifts):
    return ((uplifts['impression'].min(axis=1) >= -tol)
            & (uplifts['conversion'].min(axis=1) >= -tol))

eligible = funnel_mask(uplifts)

## I. Robust selection
Maximin revenue; tie-break bằng median. Baseline fallback nếu không có eligible grid candidate.

In [ ]:
def rank_candidates(allocations, uplifts):
    mask = funnel_mask(uplifts)
    revenue = uplifts['revenue'][mask]
    table = pd.DataFrame(allocations[mask], columns=platforms)
    table['worst_revenue_uplift'] = revenue.min(axis=1)
    table['median_revenue_uplift'] = np.median(revenue, axis=1)
    table['best_revenue_uplift'] = revenue.max(axis=1)
    table['historical_win_rate'] = (revenue > 0).mean(axis=1)
    table['worst_impression_uplift'] = uplifts['impression'][mask].min(axis=1)
    table['worst_conversion_uplift'] = uplifts['conversion'][mask].min(axis=1)
    return table.sort_values(['worst_revenue_uplift', 'median_revenue_uplift'], ascending=False)

ranked = rank_candidates(candidates, uplifts)
found = not ranked.empty
candidate = ranked.iloc[0][platforms].to_numpy() if found else x0.copy()
print('Top 10 eligible grid candidates (uplifts are fractions):')
display(ranked.head(10))
if not found:
    print('No eligible grid candidate: retain baseline.')

## J. Moved budget
Chỉ report mức reallocation; không đặt RiskLimit.

In [ ]:
moved_share = 0.5 * np.abs(candidate - x0).sum()
moved_budget = B * moved_share

## K. Expanding-window OOT
Train Jan–Jun → Jul, …, Jan–Nov → Dec. Lock allocation trước test; đây là simulation, không phải causal uplift.
Benchmarks dùng cùng baseline và test economics; best/worst chỉ xác định từ pooled training ROAS. Matched-risk dùng 0–floor(DSS moved share × 100) pp và cùng support/funnel constraints. Simple 5pp giữ nguyên dù vi phạm boundary. Uplift là fraction; uplift per 1pp = uplift / (moved share × 100).

In [ ]:
folds = []
benchmark_rows = []

def moved(allocation, baseline):
    return 0.5 * np.abs(allocation - baseline).sum()

def per_pp(uplift, moved_share):
    return uplift / (moved_share * 100) if moved_share > tol else np.nan
for test_position in range(6, 12):
    train_months = spend.index[:test_position]
    test_month = spend.index[test_position]
    baseline, budget = baseline_for(train_months)
    support = build_hull(train_months)
    train_candidates = grid[inside_historical_support(grid[:, 0], grid[:, 1], support)]
    _, train_uplifts = evaluate(train_candidates, train_months, baseline, budget)
    ranking = rank_candidates(train_candidates, train_uplifts)
    allocation = ranking.iloc[0][platforms].to_numpy() if not ranking.empty else baseline.copy()
    _, test_uplifts = evaluate(allocation[None, :], [test_month], baseline, budget)
    row = {'test_month': str(test_month), 'robust_candidate_found': not ranking.empty}
    row.update({f'baseline_{p}_share': baseline[i] for i, p in enumerate(platforms)})
    row.update({f'candidate_{p}_share': allocation[i] for i, p in enumerate(platforms)})
    row.update({f'actual_{metric}_uplift': values.item() for metric, values in test_uplifts.items()})
    folds.append(row)

    # Best/worst direction uses only pooled training economics.
    pooled = monthly.loc[(train_months, slice(None)), ['Revenue', 'Spend']].groupby('platform').sum()
    train_roas = (pooled['Revenue'] / pooled['Spend']).reindex(platforms).to_numpy()
    best, worst = train_roas.argmax(), train_roas.argmin()
    direction = np.zeros(3)
    if best != worst:
        direction[best], direction[worst] = 1, -1
    dss_moved = moved(allocation, baseline)
    previous = actual_shares.loc[train_months[-1]].to_numpy()
    simple5 = baseline + 0.05 * direction
    _, simple5_train = evaluate(simple5[None, :], train_months, baseline, budget)
    simple5_support = bool((simple5 >= 0).all() and inside_historical_support(*simple5[:2], support))
    simple5_funnel = bool(funnel_mask(simple5_train)[0])

    steps = np.arange(int(np.floor(dss_moved * 100 + tol)) + 1) / 100
    directional = baseline + steps[:, None] * direction
    supported = ((directional >= 0).all(axis=1)
                 & inside_historical_support(directional[:, 0], directional[:, 1], support))
    directional = directional[supported]
    _, directional_uplifts = evaluate(directional, train_months, baseline, budget)
    matched_ranking = rank_candidates(directional, directional_uplifts)
    matched = (matched_ranking.iloc[0][platforms].to_numpy() if not matched_ranking.empty
               else np.full(3, np.nan))

    comparison = {'test_month': str(test_month),
                  'simple5_inside_support': simple5_support,
                  'simple5_pass_funnel': simple5_funnel,
                  'matched_simple_found': not matched_ranking.empty,
                  'simple5_allocation': dict(zip(platforms, simple5))}
    for label, mix in [('dss', allocation), ('prev', previous),
                       ('simple5', simple5), ('matched_simple', matched)]:
        _, test_result = evaluate(mix[None, :], [test_month], baseline, budget)
        uplift = test_result['revenue'].item()
        movement = moved(mix, baseline)
        comparison.update({f'{label}_revenue_uplift': uplift,
                           f'{label}_moved_share': movement,
                           f'{label}_uplift_per_1pp_moved': per_pp(uplift, movement)})
        if label == 'matched_simple':
            comparison.update({f'matched_simple_{p}_share': mix[i] for i, p in enumerate(platforms)})
    benchmark_rows.append(comparison)

oot_validation = pd.DataFrame(folds)
oot_revenue = oot_validation['actual_revenue_uplift']
oot_win_rate = (oot_revenue > 0).mean()
print(f'OOT Revenue Win Rate: {oot_win_rate:.1%}')
print(f'Mean / Median / Worst OOT Revenue Uplift: {oot_revenue.mean():.4%} / '
      f'{oot_revenue.median():.4%} / {oot_revenue.min():.4%}')

oot_benchmark_comparison = pd.DataFrame(benchmark_rows)
dss_oot = oot_benchmark_comparison['dss_revenue_uplift']
matched_oot = oot_benchmark_comparison['matched_simple_revenue_uplift']
paired = matched_oot.notna() & dss_oot.notna()
difference = dss_oot[paired] - matched_oot[paired]
oot_benchmark_summary = pd.DataFrame([{
    'DSS mean uplift': dss_oot.mean(), 'DSS median uplift': dss_oot.median(),
    'DSS worst uplift': dss_oot.min(), 'Matched-simple mean uplift': matched_oot.mean(),
    'Matched-simple median uplift': matched_oot.median(), 'Matched-simple worst uplift': matched_oot.min(),
    'Paired folds': int(paired.sum()), 'Folds DSS > matched-simple': int((difference > tol).sum()),
    'Mean uplift difference': difference.mean(),
    'DSS mean moved share': oot_benchmark_comparison['dss_moved_share'].mean(),
    'Matched-simple mean moved share': oot_benchmark_comparison['matched_simple_moved_share'].mean()}])


## L. Break-even scale risk
Q4 pooled economics; deterioration đồng thời trên platform scale up. Sensitivity threshold, không phải forecast.

In [ ]:
q4 = monthly.loc[(slice(pd.Period('2024-10'), pd.Period('2024-12')), slice(None)), :]
reference = economics(q4.groupby('platform')[list(metrics.values())].sum()).reindex(platforms)
rates = reference['Rev_per_dollar'].to_numpy()
scaled_up = candidate > x0 + tol
reference_gain = candidate @ rates - x0 @ rates
scaled_revenue = candidate[scaled_up] @ rates[scaled_up]
break_even_pct = (100 * reference_gain / scaled_revenue
                  if reference_gain > tol and scaled_revenue > 0 else np.nan)

## M. Full-sample candidate
Full-sample candidate là descriptive POC result. Bằng chứng chính để đánh giá generalization phải đến từ OOT; đây chưa phải production decision.

In [ ]:
_, final_uplifts = evaluate(candidate[None, :], spend.index, x0, B)
r = final_uplifts['revenue'][0]
final_row = {f'Baseline {p} share': x0[i] for i, p in enumerate(platforms)}
final_row.update({f'Candidate {p} share': candidate[i] for i, p in enumerate(platforms)})
final_row.update({'MovedShare': moved_share, 'MovedBudget': moved_budget,
                 'Worst Revenue Uplift': r.min(), 'Median Revenue Uplift': np.median(r),
                 'Best Revenue Uplift': r.max(), 'Historical Win Rate': (r > 0).mean(),
                 'Worst Impression Uplift': final_uplifts['impression'].min(),
                 'Worst Conversion Uplift': final_uplifts['conversion'].min(),
                 'Break-even Deterioration %': break_even_pct})
full_sample_candidate = pd.DataFrame([final_row])
pipeline_summary = pd.DataFrame([{
    'Total candidate grid': len(grid), 'Candidates inside historical support': len(candidates),
    'Candidates passing funnel constraints': int(eligible.sum()),
    'Robust candidate found?': 'Yes' if found else 'No',
    'OOT Revenue Win Rate': oot_win_rate, 'Mean OOT Revenue Uplift': oot_revenue.mean(),
    'Median OOT Revenue Uplift': oot_revenue.median(), 'Worst OOT Revenue Uplift': oot_revenue.min()}])


## N. Slice consistency
Pooled ROAS theo campaign type, industry và country. Missing-platform slices không được tính là thỏa thứ tự. Đây là kiểm tra mô tả, không loại trừ mix/confounding.

In [ ]:
slice_rows = []
for dimension in ['campaign_type', 'industry', 'country']:
    totals = df.groupby([dimension, 'platform'])[['revenue', 'ad_spend']].sum()
    roas = (totals['revenue'] / totals['ad_spend']).unstack('platform').reindex(columns=platforms)
    ordered = (roas['TikTok'] > roas['Meta']) & (roas['Meta'] > roas['Google'])
    slice_rows.append({'dimension': dimension, 'total_slices': len(roas),
                       'TikTok_Meta_Google_order_count': int(ordered.sum()),
                       'consistency_rate': ordered.mean()})
slice_consistency = pd.DataFrame(slice_rows)

## O. Diminishing-return diagnostic
These are observational row-level associations, NOT causal estimates of platform-level marginal returns. Chỉ dùng rows có spend/revenue dương để lấy log; Spearman dùng pooled-form row ratios. Không dùng beta để forecast.

In [ ]:
from scipy.stats import linregress, spearmanr

diagnostic_rows = []
for platform, rows in df.groupby('platform'):
    positive = rows.loc[rows['ad_spend'].gt(0) & rows['revenue'].gt(0)]
    fit = linregress(np.log(positive['ad_spend']), np.log(positive['revenue']))
    row_roas = rows['revenue'] / rows['ad_spend'].replace(0, np.nan)
    row_cpc = rows['ad_spend'] / rows['clicks'].replace(0, np.nan)
    diagnostic_rows.append({'platform': platform, 'beta': fit.slope,
                            'beta_standard_error': fit.stderr,
                            'Spearman(ad_spend, ROAS)': spearmanr(rows['ad_spend'], row_roas, nan_policy='omit').statistic,
                            'Spearman(ad_spend, CPC)': spearmanr(rows['ad_spend'], row_cpc, nan_policy='omit').statistic})
diminishing_return_diagnostic = pd.DataFrame(diagnostic_rows).set_index('platform').reindex(platforms).reset_index()

## P. Illustrative concavity sensitivity
beta=1 corresponds to current linear assumption. beta<1 is only an illustrative concavity stress; this is NOT an estimated diminishing-return curve. Same beta across platforms is a simplifying sensitivity assumption. Các beta stress không lấy từ diagnostic; không tính break-even beta.

In [ ]:
S0, S1 = B * x0, B * candidate
reference_revenue = S0 * rates
baseline_revenue = reference_revenue.sum()
sensitivity_rows = []
for beta in [1.0, 0.9, 0.8, 0.7]:
    candidate_revenue = (reference_revenue * (S1 / S0) ** beta).sum()
    sensitivity_rows.append({'beta': beta, 'candidate_revenue': candidate_revenue,
                             'baseline_revenue': baseline_revenue,
                             'revenue_uplift': candidate_revenue / baseline_revenue - 1})
beta_sensitivity = pd.DataFrame(sensitivity_rows)
break_even_deterioration = pd.DataFrame([{'break_even_deterioration_pct': break_even_pct}])

## Q. Outputs and interpretation
Uplifts/shares là fractions, break-even là %. Paired comparison bỏ fold không có matched candidate và report số fold. “Gần bằng” dùng ngưỡng mô tả 0,1 điểm phần trăm mean uplift; không phải kiểm định thống kê. Không bootstrap: 12 states quá ít và row bootstrap không giải quyết scale-response uncertainty.

In [ ]:
for name in ['platform_summary', 'slice_consistency', 'full_sample_candidate',
             'oot_benchmark_comparison', 'oot_benchmark_summary',
             'diminishing_return_diagnostic', 'beta_sensitivity', 'break_even_deterioration']:
    print(name)
    display(globals()[name])

consistent_value = (paired.sum() == 6 and (difference > tol).all() and difference.mean() > 0.001)
print('A. PIPELINE FEASIBILITY: Core pipeline chạy end-to-end; eligible grid candidates:', int(eligible.sum()))
if consistent_value:
    print('B. SELECTOR VALUE: Pipeline is feasible and the allocation-selection layer shows incremental empirical value within the current evidence boundary.')
else:
    print('B. SELECTOR VALUE: Pipeline is feasible, but incremental value of the maximin selector is limited.')
print(f'DSS thắng {int((difference > tol).sum())}/{int(paired.sum())} paired folds; mean uplift difference: {difference.mean():.3%}.')
if paired.any() and difference.mean() <= 0.001:
    print('Simple matched-risk rule gần bằng hoặc tốt hơn DSS; data chưa cho thấy selector tạo thêm nhiều value.')
movement_gap = (oot_benchmark_comparison.loc[paired, 'dss_moved_share']
                - oot_benchmark_comparison.loc[paired, 'matched_simple_moved_share']).mean()
print(f'Mean DSS minus matched-simple moved share: {movement_gap:.3%}; chênh lệch uplift chưa tách được ảnh hưởng của lượng budget thực sự dịch chuyển.')
print('C. MODEL ROBUSTNESS: ' + ('Uplift vẫn positive ở mọi beta 1.0–0.7.'
      if beta_sensitivity['revenue_uplift'].gt(0).all() else 'Uplift không positive ở mọi beta 1.0–0.7.'))
print('OOT và beta stress là simulation/sensitivity; slice và row diagnostic chỉ là observational associations.')